In [ ]:
import os
import re
import json
import time
import copy
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    confusion_matrix,
    classification_report,
)

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, ConcatDataset
from torchvision import datasets, transforms, models

In [ ]:
# =========================================================
# 1. CONFIG
# =========================================================
SEED = 42
IMG_SIZE = 224
BATCH_SIZE = 16
NUM_EPOCHS = 30          # total target epoch count
LR = 1e-3                # better starting point for scratch training
WEIGHT_DECAY = 1e-4
NUM_WORKERS = 2

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

CLASSES = [
    "Bacterial Spot",
    "Early Blight",
    "Healthy",
    "Late Blight",
    "Septoria Leaf Spot",
    "Yellow Leaf Curl Virus",
]

SAVE_DIR = "/kaggle/working/outputs_scratch"
CKPT_DIR = os.path.join(SAVE_DIR, "checkpoints")
PLOTS_DIR = os.path.join(SAVE_DIR, "plots")
REPORTS_DIR = os.path.join(SAVE_DIR, "reports")

os.makedirs(SAVE_DIR, exist_ok=True)
os.makedirs(CKPT_DIR, exist_ok=True)
os.makedirs(PLOTS_DIR, exist_ok=True)
os.makedirs(REPORTS_DIR, exist_ok=True)

print("Device:", DEVICE)

In [ ]:
# =========================================================
# 2. DATA PATHS
# =========================================================
NEW_PLANT_ROOT = "/kaggle/input/datasets/mustainbillahtaj/new-plant-preprocessed/new_plant_preprocessed"
PLANT_DOC_ROOT = "/kaggle/input/datasets/mustainbillahtaj/plant-doc-preprocessed/processed"
PLANT_VILLAGE_ROOT = "/kaggle/input/datasets/mustainbillahtaj/plant-village-preprocessed/plant_village_preprocessed"

# Dataset 1
NP_TRAIN = f"{NEW_PLANT_ROOT}/train"
NP_VAL   = f"{NEW_PLANT_ROOT}/val"
NP_TEST  = f"{NEW_PLANT_ROOT}/test"

# Dataset 2
PD_TRAIN = f"{PLANT_DOC_ROOT}/train"
PD_VAL   = f"{PLANT_DOC_ROOT}/val"
PD_TEST  = f"{PLANT_DOC_ROOT}/test"

# Dataset 3
PV_TRAIN = f"{PLANT_VILLAGE_ROOT}/Train"
PV_VAL   = f"{PLANT_VILLAGE_ROOT}/Val"
PV_TEST  = f"{PLANT_VILLAGE_ROOT}/Test"

In [ ]:
# =========================================================
# 3. REPRODUCIBILITY
# =========================================================
def set_seed(seed: int = 42) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(SEED)

In [ ]:
# =========================================================
# 4. TRANSFORMS
# =========================================================
train_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(10),
    transforms.ColorJitter(brightness=0.15, contrast=0.15, saturation=0.15),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225]),
])

eval_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225]),
])


In [ ]:
# =========================================================
# 5. FIXED CLASS ORDER
# =========================================================
class FixedImageFolder(datasets.ImageFolder):
    def find_classes(self, directory):
        classes = CLASSES
        class_to_idx = {cls_name: i for i, cls_name in enumerate(classes)}
        return classes, class_to_idx

In [ ]:
# =========================================================
# 6. LOAD DATASETS
# =========================================================
train_datasets = [
    FixedImageFolder(NP_TRAIN, transform=train_transform),
    FixedImageFolder(PD_TRAIN, transform=train_transform),
    FixedImageFolder(PV_TRAIN, transform=train_transform),
]

val_datasets = [
    FixedImageFolder(NP_VAL, transform=eval_transform),
    FixedImageFolder(PD_VAL, transform=eval_transform),
    FixedImageFolder(PV_VAL, transform=eval_transform),
]

test_datasets = [
    FixedImageFolder(NP_TEST, transform=eval_transform),
    FixedImageFolder(PD_TEST, transform=eval_transform),
    FixedImageFolder(PV_TEST, transform=eval_transform),
]

train_dataset = ConcatDataset(train_datasets)
val_dataset = ConcatDataset(val_datasets)
test_dataset = ConcatDataset(test_datasets)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=True,
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=True,
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=True,
)

print("Train size:", len(train_dataset))
print("Val size  :", len(val_dataset))
print("Test size :", len(test_dataset))
print("Classes   :", CLASSES)

In [ ]:
# =========================================================
# 7. MODEL FROM SCRATCH
# =========================================================
model = models.efficientnet_b0(weights=None)   # FROM SCRATCH
in_features = model.classifier[1].in_features
model.classifier[1] = nn.Linear(in_features, len(CLASSES))
model = model.to(DEVICE)

criterion = nn.CrossEntropyLoss()
optimizer = optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode="min", factor=0.5, patience=3
)

In [ ]:
# =========================================================
# 8. HELPERS
# =========================================================
def save_json(data, path):
    with open(path, "w", encoding="utf-8") as f:
        json.dump(data, f, indent=2)

def compute_metrics(y_true, y_pred):
    acc = accuracy_score(y_true, y_pred)
    precision, recall, f1, _ = precision_recall_fscore_support(
        y_true, y_pred, average="macro", zero_division=0
    )
    return acc, precision, recall, f1

def run_one_epoch(model, loader, criterion, device, optimizer=None):
    is_train = optimizer is not None
    model.train() if is_train else model.eval()

    running_loss = 0.0
    all_labels = []
    all_preds = []

    for images, labels in loader:
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        if is_train:
            optimizer.zero_grad()

        with torch.set_grad_enabled(is_train):
            outputs = model(images)
            loss = criterion(outputs, labels)
            preds = torch.argmax(outputs, dim=1)

            if is_train:
                loss.backward()
                optimizer.step()

        running_loss += loss.item() * images.size(0)
        all_labels.extend(labels.detach().cpu().numpy())
        all_preds.extend(preds.detach().cpu().numpy())

    epoch_loss = running_loss / len(loader.dataset)
    acc, precision, recall, f1 = compute_metrics(all_labels, all_preds)

    return {
        "loss": float(epoch_loss),
        "accuracy": float(acc),
        "precision_macro": float(precision),
        "recall_macro": float(recall),
        "f1_macro": float(f1),
        "labels": all_labels,
        "preds": all_preds,
    }

def plot_and_save_history(history_df):
    # loss
    if {"epoch", "train_loss", "val_loss"}.issubset(history_df.columns):
        plt.figure(figsize=(8, 5))
        plt.plot(history_df["epoch"], history_df["train_loss"], label="Train Loss")
        plt.plot(history_df["epoch"], history_df["val_loss"], label="Val Loss")
        plt.xlabel("Epoch")
        plt.ylabel("Loss")
        plt.title("Loss Curve")
        plt.legend()
        plt.tight_layout()
        plt.savefig(os.path.join(PLOTS_DIR, "loss_curve.png"), dpi=200)
        plt.close()

    # accuracy
    if {"epoch", "train_accuracy", "val_accuracy"}.issubset(history_df.columns):
        plt.figure(figsize=(8, 5))
        plt.plot(history_df["epoch"], history_df["train_accuracy"], label="Train Accuracy")
        plt.plot(history_df["epoch"], history_df["val_accuracy"], label="Val Accuracy")
        plt.xlabel("Epoch")
        plt.ylabel("Accuracy")
        plt.title("Accuracy Curve")
        plt.legend()
        plt.tight_layout()
        plt.savefig(os.path.join(PLOTS_DIR, "accuracy_curve.png"), dpi=200)
        plt.close()

    # f1
    if {"epoch", "train_f1_macro", "val_f1_macro"}.issubset(history_df.columns):
        plt.figure(figsize=(8, 5))
        plt.plot(history_df["epoch"], history_df["train_f1_macro"], label="Train F1")
        plt.plot(history_df["epoch"], history_df["val_f1_macro"], label="Val F1")
        plt.xlabel("Epoch")
        plt.ylabel("Macro F1")
        plt.title("F1 Curve")
        plt.legend()
        plt.tight_layout()
        plt.savefig(os.path.join(PLOTS_DIR, "f1_curve.png"), dpi=200)
        plt.close()

def save_history(history):
    history_df = pd.DataFrame(history)
    history_df.to_csv(os.path.join(SAVE_DIR, "training_history.csv"), index=False)
    save_json(history, os.path.join(SAVE_DIR, "training_history.json"))
    plot_and_save_history(history_df)
    return history_df

def save_epoch_checkpoint(epoch, model, optimizer, scheduler, history, best_val_f1, best_val_acc):
    payload = {
        "epoch": epoch,
        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "scheduler_state_dict": scheduler.state_dict(),
        "history": history,
        "best_val_f1": best_val_f1,
        "best_val_acc": best_val_acc,
        "classes": CLASSES,
        "img_size": IMG_SIZE,
        "batch_size": BATCH_SIZE,
        "learning_rate": LR,
        "from_scratch": True,
    }

    # Save per-epoch checkpoint
    epoch_path = os.path.join(CKPT_DIR, f"checkpoint_epoch_{epoch:02d}.pth")
    torch.save(payload, epoch_path)

    # Save latest checkpoint too
    torch.save(payload, os.path.join(SAVE_DIR, "latest_checkpoint.pth"))

def save_best_model(epoch, model, optimizer, scheduler, best_val_f1):
    payload = {
        "epoch": epoch,
        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "scheduler_state_dict": scheduler.state_dict(),
        "best_val_f1": best_val_f1,
        "classes": CLASSES,
        "from_scratch": True,
    }
    torch.save(payload, os.path.join(SAVE_DIR, "best_model.pth"))

def save_crash_checkpoint(epoch, model, optimizer, scheduler, history, best_val_f1, best_val_acc, error_text):
    payload = {
        "epoch": epoch,
        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "scheduler_state_dict": scheduler.state_dict(),
        "history": history,
        "best_val_f1": best_val_f1,
        "best_val_acc": best_val_acc,
        "classes": CLASSES,
        "error": error_text,
        "from_scratch": True,
    }
    torch.save(payload, os.path.join(SAVE_DIR, "crash_checkpoint.pth"))

def find_latest_checkpoint(ckpt_dir):
    if not os.path.exists(ckpt_dir):
        return None

    ckpt_files = [
        f for f in os.listdir(ckpt_dir)
        if f.startswith("checkpoint_epoch_") and f.endswith(".pth")
    ]

    if not ckpt_files:
        return None

    def extract_epoch(fname):
        m = re.search(r"checkpoint_epoch_(\d+)\.pth", fname)
        return int(m.group(1)) if m else -1

    ckpt_files = sorted(ckpt_files, key=extract_epoch)
    return os.path.join(ckpt_dir, ckpt_files[-1])


In [ ]:
# =========================================================
# 9. RESUME LOGIC
# =========================================================
start_epoch = 0
history = []
best_val_f1 = -1.0
best_val_acc = -1.0

latest_ckpt = find_latest_checkpoint(CKPT_DIR)

if latest_ckpt is not None:
    print(f"Resuming from checkpoint: {latest_ckpt}")
    checkpoint = torch.load(latest_ckpt, map_location=DEVICE)

    model.load_state_dict(checkpoint["model_state_dict"])
    optimizer.load_state_dict(checkpoint["optimizer_state_dict"])
    scheduler.load_state_dict(checkpoint["scheduler_state_dict"])

    start_epoch = checkpoint.get("epoch", 0)
    history = checkpoint.get("history", [])
    best_val_f1 = checkpoint.get("best_val_f1", -1.0)
    best_val_acc = checkpoint.get("best_val_acc", -1.0)

    print(f"Resumed from epoch {start_epoch}")
    print(f"Best val F1 so far : {best_val_f1:.4f}")
    print(f"Best val Acc so far: {best_val_acc:.4f}")
else:
    print("No checkpoint found. Starting from scratch.")

In [ ]:
# =========================================================
# 10. TRAINING LOOP
# =========================================================
# Note:
# If you already finished 30 epochs and want 10 more,
# set NUM_EPOCHS = 40.

best_model_wts = copy.deepcopy(model.state_dict())

try:
    for epoch in range(start_epoch, NUM_EPOCHS):
        epoch_start = time.time()

        train_out = run_one_epoch(model, train_loader, criterion, DEVICE, optimizer=optimizer)
        val_out = run_one_epoch(model, val_loader, criterion, DEVICE, optimizer=None)

        scheduler.step(val_out["loss"])
        current_lr = optimizer.param_groups[0]["lr"]
        epoch_time = time.time() - epoch_start

        record = {
            "epoch": epoch + 1,
            "train_loss": train_out["loss"],
            "train_accuracy": train_out["accuracy"],
            "train_precision_macro": train_out["precision_macro"],
            "train_recall_macro": train_out["recall_macro"],
            "train_f1_macro": train_out["f1_macro"],
            "val_loss": val_out["loss"],
            "val_accuracy": val_out["accuracy"],
            "val_precision_macro": val_out["precision_macro"],
            "val_recall_macro": val_out["recall_macro"],
            "val_f1_macro": val_out["f1_macro"],
            "lr": current_lr,
            "epoch_time_sec": epoch_time,
        }
        history.append(record)

        # Update best model
        if val_out["f1_macro"] > best_val_f1:
            best_val_f1 = val_out["f1_macro"]
            best_model_wts = copy.deepcopy(model.state_dict())
            save_best_model(epoch + 1, model, optimizer, scheduler, best_val_f1)

        if val_out["accuracy"] > best_val_acc:
            best_val_acc = val_out["accuracy"]

        # Save history and checkpoint every epoch
        save_history(history)
        save_epoch_checkpoint(
            epoch=epoch + 1,
            model=model,
            optimizer=optimizer,
            scheduler=scheduler,
            history=history,
            best_val_f1=best_val_f1,
            best_val_acc=best_val_acc,
        )

        print(
            f"Epoch [{epoch+1}/{NUM_EPOCHS}] | "
            f"Train Loss: {train_out['loss']:.4f}, "
            f"Train Acc: {train_out['accuracy']:.4f}, "
            f"Train F1: {train_out['f1_macro']:.4f} | "
            f"Val Loss: {val_out['loss']:.4f}, "
            f"Val Acc: {val_out['accuracy']:.4f}, "
            f"Val F1: {val_out['f1_macro']:.4f} | "
            f"LR: {current_lr:.6f} | "
            f"Time: {epoch_time:.1f}s"
        )

except Exception as e:
    error_text = str(e)
    print("Training interrupted by error:")
    print(error_text)

    save_crash_checkpoint(
        epoch=epoch + 1 if "epoch" in locals() else start_epoch,
        model=model,
        optimizer=optimizer,
        scheduler=scheduler,
        history=history,
        best_val_f1=best_val_f1,
        best_val_acc=best_val_acc,
        error_text=error_text,
    )

    if len(history) > 0:
        save_history(history)

    raise


In [ ]:
# =========================================================
# 11. LOAD BEST MODEL FOR FINAL EVALUATION
# =========================================================
best_model_path = os.path.join(SAVE_DIR, "best_model.pth")
if os.path.exists(best_model_path):
    best_ckpt = torch.load(best_model_path, map_location=DEVICE)
    model.load_state_dict(best_ckpt["model_state_dict"])
    print(f"Loaded best model from epoch {best_ckpt.get('epoch', 'unknown')}")
else:
    model.load_state_dict(best_model_wts)
    print("best_model.pth not found, using in-memory best weights.")

In [ ]:
# =========================================================
# 12. FINAL TEST EVALUATION
# =========================================================
test_out = run_one_epoch(model, test_loader, criterion, DEVICE, optimizer=None)

test_loss = test_out["loss"]
test_acc = test_out["accuracy"]
test_prec = test_out["precision_macro"]
test_rec = test_out["recall_macro"]
test_f1 = test_out["f1_macro"]
test_labels = test_out["labels"]
test_preds = test_out["preds"]

print("\nFINAL TEST RESULTS")
print(f"Test Loss           : {test_loss:.4f}")
print(f"Test Accuracy       : {test_acc:.4f}")
print(f"Test PrecisionMacro : {test_prec:.4f}")
print(f"Test RecallMacro    : {test_rec:.4f}")
print(f"Test F1Macro        : {test_f1:.4f}")

# Save final metrics
final_metrics = {
    "test_loss": test_loss,
    "test_accuracy": test_acc,
    "test_precision_macro": test_prec,
    "test_recall_macro": test_rec,
    "test_f1_macro": test_f1,
    "from_scratch": True,
    "model": "efficientnet_b0",
    "img_size": IMG_SIZE,
}
save_json(final_metrics, os.path.join(SAVE_DIR, "final_test_metrics.json"))


In [ ]:
# =========================================================
# 13. CLASSIFICATION REPORT
# =========================================================
report_text = classification_report(
    test_labels,
    test_preds,
    target_names=CLASSES,
    digits=4,
    zero_division=0,
)

with open(os.path.join(REPORTS_DIR, "classification_report.txt"), "w", encoding="utf-8") as f:
    f.write(report_text)

report_dict = classification_report(
    test_labels,
    test_preds,
    target_names=CLASSES,
    digits=4,
    zero_division=0,
    output_dict=True,
)
save_json(report_dict, os.path.join(REPORTS_DIR, "classification_report.json"))

print("\nClassification report:")
print(report_text)


In [ ]:
# =========================================================
# 14. CONFUSION MATRIX
# =========================================================
cm = confusion_matrix(test_labels, test_preds)
np.save(os.path.join(REPORTS_DIR, "confusion_matrix.npy"), cm)

plt.figure(figsize=(8, 6))
plt.imshow(cm, interpolation="nearest")
plt.title("Confusion Matrix")
plt.colorbar()

tick_marks = np.arange(len(CLASSES))
plt.xticks(tick_marks, CLASSES, rotation=45, ha="right")
plt.yticks(tick_marks, CLASSES)

for i in range(cm.shape[0]):
    for j in range(cm.shape[1]):
        plt.text(j, i, format(cm[i, j], "d"), ha="center", va="center")

plt.ylabel("True Label")
plt.xlabel("Predicted Label")
plt.tight_layout()
plt.savefig(os.path.join(REPORTS_DIR, "confusion_matrix.png"), dpi=200, bbox_inches="tight")
plt.show()

In [ ]:
# =========================================================
# 15. INFERENCE TIME
# =========================================================
def measure_inference_time(model, loader, device, num_batches=20):
    model.eval()
    total_time = 0.0
    total_images = 0
    batch_times = []

    with torch.no_grad():
        for batch_idx, (images, _) in enumerate(loader):
            if batch_idx >= num_batches:
                break

            images = images.to(device, non_blocking=True)

            if device.type == "cuda":
                torch.cuda.synchronize()

            start = time.perf_counter()
            _ = model(images)

            if device.type == "cuda":
                torch.cuda.synchronize()

            end = time.perf_counter()

            batch_time = end - start
            total_time += batch_time
            total_images += images.size(0)
            batch_times.append(batch_time)

    avg_time_per_batch = total_time / len(batch_times) if batch_times else None
    avg_time_per_image = total_time / total_images if total_images > 0 else None
    images_per_second = total_images / total_time if total_time > 0 else None

    return {
        "num_batches_measured": len(batch_times),
        "total_images_measured": total_images,
        "total_inference_time_sec": total_time,
        "avg_time_per_batch_sec": avg_time_per_batch,
        "avg_time_per_image_sec": avg_time_per_image,
        "images_per_second": images_per_second,
    }

inference_stats = measure_inference_time(model, test_loader, DEVICE, num_batches=20)
save_json(inference_stats, os.path.join(SAVE_DIR, "inference_time.json"))

print("\nINFERENCE TIME")
for k, v in inference_stats.items():
    print(f"{k}: {v}")


In [ ]:
# =========================================================
# 16. SAVE FILE SUMMARY
# =========================================================
summary = {}
for root, dirs, files in os.walk(SAVE_DIR):
    rel_root = os.path.relpath(root, SAVE_DIR)
    summary[rel_root] = sorted(files)

save_json(summary, os.path.join(SAVE_DIR, "saved_files_summary.json"))

print("\nAll outputs saved under:", SAVE_DIR)

In [ ]:
# =========================================================
# 17. OPTIONAL: PRINT SAVED FILES
# =========================================================
print("\nSaved files:")
for root, dirs, files in os.walk(SAVE_DIR):
    level = root.replace(SAVE_DIR, "").count(os.sep)
    indent = "    " * level
    print(f"{indent}{os.path.basename(root)}/")
    subindent = "    " * (level + 1)
    for f in sorted(files):
        print(f"{subindent}{f}")

In [ ]:
import shutil
import os

root_dir = "/kaggle/working"
base_dir = "outputs_scratch"

zip_path = shutil.make_archive(
    base_name="/kaggle/working/outputs_scratch",
    format="zip",
    root_dir=root_dir,
    base_dir=base_dir,
)

print("Created:", zip_path)
print("Exists:", os.path.exists(zip_path))
print("Size (MB):", round(os.path.getsize(zip_path) / (1024 * 1024), 2))

In [ ]:
import zipfile

zip_file = "/kaggle/working/outputs_scratch.zip"

with zipfile.ZipFile(zip_file, "r") as z:
    names = z.namelist()
    print("Total files in zip:", len(names))
    print("\nFirst 30 entries:")
    for name in names[:30]:
        print(name)

In [ ]:
import os
import zipfile

zip_path = "/kaggle/working/outputs_scratch_selected.zip"
save_dir = "/kaggle/working/outputs_scratch"

files_to_add = [
    "best_model.pth",
    "final_test_metrics.json",
    "inference_time.json",
    "training_history.csv",
    "training_history.json",
    "saved_files_summary.json",
]

with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as z:
    for fname in files_to_add:
        fpath = os.path.join(save_dir, fname)
        if os.path.exists(fpath):
            z.write(fpath, arcname=f"outputs_scratch/{fname}")

    ckpt_dir = os.path.join(save_dir, "checkpoints")
    if os.path.exists(ckpt_dir):
        for f in sorted(os.listdir(ckpt_dir)):
            full = os.path.join(ckpt_dir, f)
            if os.path.isfile(full):
                z.write(full, arcname=f"outputs_scratch/checkpoints/{f}")

    reports_dir = os.path.join(save_dir, "reports")
    if os.path.exists(reports_dir):
        for f in sorted(os.listdir(reports_dir)):
            full = os.path.join(reports_dir, f)
            if os.path.isfile(full):
                z.write(full, arcname=f"outputs_scratch/reports/{f}")

    plots_dir = os.path.join(save_dir, "plots")
    if os.path.exists(plots_dir):
        for f in sorted(os.listdir(plots_dir)):
            full = os.path.join(plots_dir, f)
            if os.path.isfile(full):
                z.write(full, arcname=f"outputs_scratch/plots/{f}")

print("Created:", zip_path)

In [ ]:
import os
import zipfile

zip_path = "/kaggle/working/outputs_scratch.zip"

print("Exists:", os.path.exists(zip_path))
print("Size (MB):", round(os.path.getsize(zip_path) / (1024 * 1024), 2))

with zipfile.ZipFile(zip_path, "r") as z:
    print("Number of files in zip:", len(z.namelist()))
    print("First 20:")
    for x in z.namelist()[:20]:
        print(x)